# 03 - Distortions (Part 2: introduce distortions, measure degradation)

Applies the 3 locked distortions (salt & pepper noise, motion blur, JPEG
compression) at every configured intensity level, re-runs all 4 tasks, and
plots performance vs. SNR - the PDF's required "range of distortion
intensities, measure as SNR" evaluation.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Runtime already had this repo cloned from an earlier cell run in this
        # session - pull so we don't keep running against a stale checkout.
        get_ipython().system(f"git -C {REPO_DIR} pull")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow
from ipproj.datasets.kitti import read_image
from ipproj.datasets.kitti_flow import read_kitti_flow_png
from ipproj.datasets.materialize import materialize_transformed
from ipproj.tasks import feature_matching, optical_flow, object_detection, semantic_segmentation
from ipproj.distortions import REGISTRY as DISTORTIONS
from ipproj.metrics.snr import compute_snr_db
from ipproj.viz.plotting import plot_before_after, plot_metric_vs_intensity, save_figure

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()
flow_splits = kitti_flow.load_optical_flow_subset()

checkpoint_path = (config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").read_text().strip()
yolo_model = YOLO(checkpoint_path)
segformer_model, segformer_processor = semantic_segmentation.load_pretrained()

## Before/after visualization per distortion (strongest intensity)

In [ ]:
sample_image = read_image(detection_splits["test"][0].image_path)
for name, module in DISTORTIONS.items():
    distorted = module.distort(sample_image, level=len(module.LEVELS) - 1)
    fig = plot_before_after(sample_image, distorted, title_after=f"{name} (max intensity)")
    save_figure(fig, f"03_distortions/before_after_{name}.png")

## Degradation sweep

For each distortion x intensity level: distort the test split, re-run all 4
tasks, and record metrics + a representative SNR value. Distorted images are
persisted under `config.DISTORTED_ROOT` (via `materialize_transformed`) so
`tasks.*.evaluate()` can be reused unchanged - it always reads from a
sample's `.image_path`.

In [ ]:
results = []

for name, module in DISTORTIONS.items():
    for level in range(len(module.LEVELS)):
        distort_fn = lambda img, m=module, l=level: m.distort(img, l)
        out_dir = config.DISTORTED_ROOT / name / f"level_{level}"

        distorted_detection = materialize_transformed(detection_splits["test"], distort_fn, out_dir / "detection")
        detection_metrics = object_detection.evaluate(yolo_model, distorted_detection)

        distorted_segmentation = materialize_transformed(segmentation_splits["test"], distort_fn, out_dir / "segmentation")
        segmentation_metrics = semantic_segmentation.evaluate(segformer_model, segformer_processor, distorted_segmentation)

        match_accuracies = []
        for sample in detection_splits["test"][:20]:
            clean = read_image(sample.image_path)
            match_accuracies.append(feature_matching.match(clean, module.distort(clean, level))[3])

        epe_values = []
        for sample in flow_splits["test"]:
            frame1 = read_image(sample.frame1_path)
            frame2 = read_image(sample.frame2_path)
            gt_flow, valid = read_kitti_flow_png(sample.flow_gt_path)
            epe_values.append(optical_flow.evaluate(frame1, module.distort(frame2, level), gt_flow, valid))

        reference_clean = read_image(detection_splits["test"][0].image_path)
        sample_snr = compute_snr_db(reference_clean, module.distort(reference_clean, level))

        results.append({
            "distortion": name,
            "level": level,
            "snr_db": sample_snr,
            "match_accuracy": float(np.mean(match_accuracies)),
            "epe": float(np.mean(epe_values)),
            "map": float(detection_metrics["map"]),
            "mean_iou": float(segmentation_metrics.mean()),
        })

distortion_results = pd.DataFrame(results)
config.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
distortion_results.to_csv(config.RESULTS_ROOT / "03_distortions.csv", index=False)
distortion_results

## Performance vs. SNR

In [ ]:
for name in DISTORTIONS:
    subset = distortion_results[distortion_results["distortion"] == name].sort_values("snr_db")
    fig = plot_metric_vs_intensity(
        subset["snr_db"].tolist(),
        {
            "mAP": subset["map"].tolist(),
            "mean IoU": subset["mean_iou"].tolist(),
            "match accuracy": subset["match_accuracy"].tolist(),
        },
        xlabel="SNR (dB)", ylabel="metric value", title=f"Performance vs SNR - {name}",
    )
    save_figure(fig, f"03_distortions/performance_vs_snr_{name}.png")